# Step 3: Chunking, Embedding & Azure AI Search Indexing

Split enriched page text into chunks, generate embeddings, and push to Azure AI Search.

In [1]:
%pip install openai azure-identity azure-search-documents python-dotenv langchain-text-splitters -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import json
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SimpleField,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv(override=True)

# Azure OpenAI client (same auth as notebook 02)
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)
openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_ad_token_provider=token_provider,
    api_version="2024-12-01-preview",
)
EMBEDDING_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Azure AI Search
SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME", "rag-index")
search_credential = DefaultAzureCredential()

# Load enriched pages from Step 2
with open("extracted_data/parsed_pages_with_images.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

##print(f"OpenAI endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Embedding model: {EMBEDDING_DEPLOYMENT}")
##print(f"Search endpoint: {SEARCH_ENDPOINT}")
print(f"Index name: {INDEX_NAME}")
print(f"Pages loaded: {len(pages)}")

c:\Projects\RAG\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Embedding model: text-embedding-3-large-460208
Index name: rag-index
Pages loaded: 3623


In [11]:
# --- Chunk the pages ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

chunks = []
for page in pages:
    text = page.get("text_with_images", page["text"])
    if not text.strip():
        continue
    
    page_chunks = splitter.split_text(text)
    for i, chunk_text in enumerate(page_chunks):
        chunks.append({
            "id": f"page-{page['page_number']}-chunk-{i}",
            "content": chunk_text,
            "page_number": page["page_number"],
            "has_images": page.get("has_images", False),
            "chunk_index": i,
        })

print(f"Total chunks created: {len(chunks)}")
print(f"Avg chunk size: {sum(len(c['content']) for c in chunks) // len(chunks)} chars")
print(f"\nSample chunk (page {chunks[5]['page_number']}, chunk {chunks[5]['chunk_index']}):")
print(chunks[5]["content"][:300])

Total chunks created: 9104
Avg chunk size: 797 chars

Sample chunk (page 9, chunk 0):
FOR JESSICA, WHO LOVES STORIES,
FOR ANNE, WHO LOVED THEM TOO;
AND FOR DI, WHO HEARD THIS ONE FIRST.


In [12]:
# --- Generate embeddings in batches (256 dims to fit free-tier storage) ---
BATCH_SIZE = 100
EMBEDDING_DIMS = 256  # Reduced from 3072 to fit Azure AI Search free tier (50MB limit)

def get_embeddings(texts: list[str]) -> list[list[float]]:
    response = openai_client.embeddings.create(
        model=EMBEDDING_DEPLOYMENT,
        input=texts,
        dimensions=EMBEDDING_DIMS,
    )
    return [item.embedding for item in response.data]

# Process all chunks
all_embeddings = []
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    batch_texts = [c["content"] for c in batch]
    embeddings = get_embeddings(batch_texts)
    all_embeddings.extend(embeddings)
    
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  Embedded {i + len(batch)}/{len(chunks)} chunks")

# Attach embeddings to chunks
for chunk, embedding in zip(chunks, all_embeddings):
    chunk["embedding"] = embedding

print(f"\nEmbeddings done: {len(all_embeddings)} vectors")
print(f"Vector dimensions: {len(all_embeddings[0])}")
print(f"Estimated vector storage: {len(all_embeddings) * EMBEDDING_DIMS * 4 / 1024 / 1024:.1f} MB")

  Embedded 100/9104 chunks
  Embedded 1100/9104 chunks
  Embedded 2100/9104 chunks
  Embedded 3100/9104 chunks
  Embedded 4100/9104 chunks
  Embedded 5100/9104 chunks
  Embedded 6100/9104 chunks
  Embedded 7100/9104 chunks
  Embedded 8100/9104 chunks
  Embedded 9100/9104 chunks

Embeddings done: 9104 vectors
Vector dimensions: 256
Estimated vector storage: 8.9 MB


In [24]:
# --- Create Azure AI Search index with semantic ranking ---
VECTOR_DIMS = len(chunks[0]["embedding"])

index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_credential)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="page_number", type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SimpleField(name="has_images", type=SearchFieldDataType.Boolean, filterable=True),
    SimpleField(name="chunk_index", type=SearchFieldDataType.Int32, filterable=True),
    SearchField(
        name="embedding",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        vector_search_dimensions=VECTOR_DIMS,
        vector_search_profile_name="default-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="default-hnsw")],
    profiles=[VectorSearchProfile(name="default-profile", algorithm_configuration_name="default-hnsw")],
)

# Semantic ranking configuration — reranks results using a Microsoft-hosted model
semantic_config = SemanticConfiguration(
    name="default-semantic",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")]
    ),
)
semantic_search = SemanticSearch(configurations=[semantic_config])

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
result = index_client.create_or_update_index(index)

print(f"Index '{result.name}' created/updated with semantic ranking")

Index 'rag-index' created/updated with semantic ranking


In [25]:
# --- Upload chunks to Azure AI Search ---
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_credential)

UPLOAD_BATCH = 500
uploaded = 0

for i in range(0, len(chunks), UPLOAD_BATCH):
    batch = chunks[i:i + UPLOAD_BATCH]
    # Prepare documents for upload
    docs = [
        {
            "id": c["id"],
            "content": c["content"],
            "page_number": c["page_number"],
            "has_images": c["has_images"],
            "chunk_index": c["chunk_index"],
            "embedding": c["embedding"],
        }
        for c in batch
    ]
    result = search_client.upload_documents(documents=docs)
    uploaded += len(batch)
    print(f"  Uploaded {uploaded}/{len(chunks)}")

print(f"\nDone! {uploaded} chunks indexed in '{INDEX_NAME}'")

  Uploaded 500/9104
  Uploaded 1000/9104
  Uploaded 1500/9104
  Uploaded 2000/9104
  Uploaded 2500/9104
  Uploaded 3000/9104
  Uploaded 3500/9104
  Uploaded 4000/9104
  Uploaded 4500/9104
  Uploaded 5000/9104
  Uploaded 5500/9104
  Uploaded 6000/9104
  Uploaded 6500/9104
  Uploaded 7000/9104
  Uploaded 7500/9104
  Uploaded 8000/9104
  Uploaded 8500/9104
  Uploaded 9000/9104
  Uploaded 9104/9104

Done! 9104 chunks indexed in 'rag-index'
